In [1]:
!pip install -q diffusers transformers accelerate
!pip install -q lpips
!pip install -q datasets
!pip install -q gradio
!pip install -q deep-translator langdetect
!pip install -q opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 33.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.3 MB/s eta 0:00:00


In [8]:
#diffuser et frontend

import os
import glob
import urllib.request
import zipfile
import re
import requests
from io import BytesIO
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
from PIL import Image
from datasets import load_dataset
import lpips
import gradio as gr
from diffusers import StableDiffusionImg2ImgPipeline
from deep_translator import GoogleTranslator
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0  # détection de langue reproductible

# =====================================================================
# 1. INITIALISATION DES DOSSIERS ET DATASETS
# =====================================================================
os.makedirs("data/dataset_1_coco", exist_ok=True)
os.makedirs("data/dataset_2_maroc", exist_ok=True)
os.makedirs("data/dataset_3_wikiart", exist_ok=True)

# Téléchargement COCO (Échantillon léger pour éviter les crashs de stockage)
coco_url = "http://images.cocodataset.org/zips/val2017.zip"
coco_zip = "coco_val2017.zip"
if not os.path.exists("data/dataset_1_coco/val2017"):
    print("1. Téléchargement de COCO en cours...")
    urllib.request.urlretrieve(coco_url, coco_zip)
    print("2. Extraction des images COCO...")
    with zipfile.ZipFile(coco_zip, 'r') as zip_ref:
        zip_ref.extractall("data/dataset_1_coco")
    os.remove(coco_zip)
    print("-> Dataset 1 (COCO) installé avec succès !")

# Téléchargement WikiArt
if len(os.listdir("data/dataset_3_wikiart")) == 0:
    print("Téléchargement d'un échantillon massif de WikiArt...")
    dataset_art = load_dataset("huggan/wikiart", split="train", streaming=True)
    compteur_art = 0
    for item in dataset_art:
        if compteur_art >= 100:
            break
        try:
            image = item['image']
            image.save(f"data/dataset_3_wikiart/art_{compteur_art}.jpg")
            compteur_art += 1
        except Exception as e:
            continue
    print(f"-> Dataset 3 (WikiArt) installé avec {compteur_art} peintures !")

# Téléchargement Patrimoine Marocain
if len(os.listdir("data/dataset_2_maroc")) == 0:
    mots_cles = [
        "morocco-architecture", "zellige", "marrakech-medina", "hassan-ii-mosque",
        "bab-boujloud", "volubilis-morocco", "moroccan-riad-courtyard", "moroccan-pattern",
        "chefchaouen", "islamic-geometric-art", "moroccan-rug-patterns",
        "moroccan-leather-tannery", "moroccan-metal-lantern"
    ]
    images_par_mot_cle = 5
    compteur = 0
    print("Téléchargement des images du Patrimoine Marocain...")
    for keyword in mots_cles:
        for i in range(images_par_mot_cle):
            try:
                url = f"https://loremflickr.com/800/800/{keyword}?random={i}"
                headers = {"User-Agent": "Mozilla/5.0"}
                response = requests.get(url, headers=headers, timeout=15)
                if response.status_code == 200:
                    img = Image.open(BytesIO(response.content))
                    chemin_sauvegarde = f"data/dataset_2_maroc/maroc_{compteur}.jpg"
                    img.convert("RGB").save(chemin_sauvegarde, "JPEG")
                    compteur += 1
            except Exception as e:
                continue
    print(f"-> Dataset 2 (Patrimoine Marocain) installé avec succès : {compteur} images !")

# =====================================================================
# 2. DEVICE & CORE NST ALGORITHM (VGG19)
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class VGG(nn.Module):
    def __init__(self):
        super(VGG, self).__init__()
        self.chosen_features = ['0', '5', '10', '19', '28']
        self.model = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features[:29].to(device).eval()
        # Le modèle ne doit jamais être entraîné : on coupe les gradients sur ses poids
        for param in self.model.parameters():
            param.requires_grad_(False)

    def forward(self, x):
        features = []
        for name, layer in self.model._modules.items():
            x = layer(x)
            if name in self.chosen_features:
                features.append(x)
        return features


# Chargement UNIQUE de VGG19 au démarrage du notebook (pas à chaque clic).
# Recharger ~80 Mo de poids à chaque génération était l'une des causes principales de lenteur.
print("Chargement unique du modèle VGG19 pour le NST...")
_vgg_model = VGG()
print("-> VGG19 chargé et mis en cache.")


def run_neural_style_transfer(content_tensor, style_tensor, num_steps, alpha=1, beta=1e6, progress=None):
    generated_tensor = content_tensor.clone().requires_grad_(True)
    optimizer = optim.Adam([generated_tensor], lr=0.01)

    # Les features de contenu et de style sont FIXES : on ne les recalcule qu'une seule fois,
    # avant la boucle, au lieu de les recalculer à chaque step (gain énorme de temps).
    with torch.no_grad():
        content_features = _vgg_model(content_tensor)
        style_features = _vgg_model(style_tensor)

    for step in range(num_steps):
        generated_features = _vgg_model(generated_tensor)

        style_loss = content_loss = 0

        for g, c, s in zip(generated_features, content_features, style_features):
            content_loss += torch.mean((g - c) ** 2)
            _, d, h, w = g.shape
            G = g.view(d, h * w).mm(g.view(d, h * w).t())
            A = s.view(d, h * w).mm(s.view(d, h * w).t())
            style_loss += torch.mean((G - A) ** 2) / (d * h * w)

        total_loss = alpha * content_loss + beta * style_loss
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        if progress is not None:
            progress((step + 1) / num_steps, desc=f"Optimisation NST — étape {step + 1}/{num_steps}")

    return generated_tensor

# =====================================================================
# 3. PIPELINE STABLE DIFFUSION (IMAGE-TO-IMAGE)
# =====================================================================
print("Initialisation du pipeline Stable Diffusion Img2Img...")
try:
    diffusion_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        safety_checker=None  # Gain de VRAM sur T4 ; remettre un safety_checker si usage public non modéré
    ).to(device)

    # Optimisations mémoire essentielles pour le GPU limité de Colab (T4)
    diffusion_pipe.enable_attention_slicing()
    diffusion_pipe.enable_vae_slicing()
    try:
        diffusion_pipe.enable_xformers_memory_efficient_attention()
        print("-> xformers activé (attention plus rapide et plus légère).")
    except Exception:
        print("-> xformers non disponible, on continue sans (pas bloquant).")

    print("-> Pipeline Stable Diffusion chargé avec succès !")
except Exception as e:
    print(f"Alerte chargement diffusion pipeline : {e}")
    diffusion_pipe = None

# =====================================================================
# 4. PRETRAITEMENT, POSTTRAITEMENT & METRIQUES
# =====================================================================
class MasterImagePreprocessor:
    def __init__(self, target_size=(512, 512), augment=False):
        self.target_size = target_size
        self.device = device
        self.imagenet_mean = [0.485, 0.456, 0.406]
        self.imagenet_std = [0.229, 0.224, 0.225]

        transform_list = [transforms.Resize(self.target_size)]
        if augment:
            transform_list.append(transforms.RandomHorizontalFlip(p=0.5))

        transform_list.extend([
            transforms.ToTensor(),
            transforms.Normalize(mean=self.imagenet_mean, std=self.imagenet_std)
        ])
        self.nst_transform = transforms.Compose(transform_list)

    def load_and_crop_center(self, image_path):
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            w, h = img.size
            min_dim = min(w, h)
            left = (w - min_dim) / 2
            top = (h - min_dim) / 2
            right = (w + min_dim) / 2
            bottom = (h + min_dim) / 2
            img_cropped = img.crop((left, top, right, bottom))
            return img_cropped.resize(self.target_size, Image.Resampling.LANCZOS)

    def preprocess_for_nst(self, image_path):
        cropped_img = self.load_and_crop_center(image_path)
        tensor_img = self.nst_transform(cropped_img)
        return tensor_img.unsqueeze(0).to(self.device)

    def postprocess_to_pil(self, tensor):
        detransform = transforms.Compose([
            transforms.Normalize(
                mean=[-m/s for m, s in zip(self.imagenet_mean, self.imagenet_std)],
                std=[1/s for s in self.imagenet_std]
            ),
            transforms.Lambda(lambda x: x.clamp(0, 1)),
            transforms.ToPILImage()
        ])
        return detransform(tensor.squeeze(0).cpu())

    def preprocess_for_diffusion(self, image_path):
        return self.load_and_crop_center(image_path)

def compute_metrics(original_pil, generated_pil):
    img1 = np.array(original_pil.convert('L'))
    img2 = np.array(generated_pil.convert('L'))
    C1 = (0.01 * 255)**2
    C2 = (0.03 * 255)**2
    mu1, mu2 = img1.mean(), img2.mean()
    sigma1_sq, sigma2_sq = img1.var(), img2.var()
    sigma12 = np.cov(img1.flatten(), img2.flatten())[0, 1]
    ssim_idx = ((2 * mu1 * mu2 + C1) * (2 * sigma12 + C2)) / ((mu1**2 + mu2**2 + C1) * (sigma1_sq + sigma2_sq + C2))
    ssim_score = max(0, min(1, ssim_idx))

    loss_fn_alex = lpips.LPIPS(net='alex')
    transform_metric = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    tensor_orig = transform_metric(original_pil).unsqueeze(0)
    tensor_gen = transform_metric(generated_pil).unsqueeze(0)
    with torch.no_grad():
        lpips_dist = loss_fn_alex(tensor_orig, tensor_gen).item()

    return round(ssim_score, 4), round(lpips_dist, 4)

# =====================================================================
# 5. SUPPORT MULTILINGUE (FR / AR / EN) POUR LES PROMPTS DE DIFFUSION
# =====================================================================
def translate_prompt_to_english(prompt):
    """
    Détecte la langue du prompt (FR, AR, EN, ...) et le traduit en anglais
    si nécessaire, car SD 1.5 (CLIP) ne comprend nativement que l'anglais.
    Léger : pas de modèle chargé en RAM/VRAM, juste un appel réseau.
    """
    if not prompt or not prompt.strip():
        return prompt, "en"

    try:
        detected_lang = detect(prompt)
    except Exception:
        detected_lang = "en"

    if detected_lang == "en":
        return prompt, "en"

    try:
        translated = GoogleTranslator(source="auto", target="en").translate(prompt)
        print(f"-> Prompt détecté en '{detected_lang}', traduit en anglais : {translated}")
        return translated, detected_lang
    except Exception as e:
        print(f"Alerte traduction : {e} -> on garde le prompt original.")
        return prompt, detected_lang


def enrich_prompt_for_quality(prompt_en):
    """
    Ajoute des 'quality boosters' standards qui améliorent nettement le rendu
    de SD 1.5 sans changer le sens du prompt utilisateur.
    """
    quality_suffix = "highly detailed, sharp focus, professional artwork, masterpiece, best quality, 4k"
    return f"{prompt_en}, {quality_suffix}"


DEFAULT_NEGATIVE_PROMPT = (
    "blurry, low quality, low resolution, deformed, disfigured, bad anatomy, "
    "extra limbs, watermark, text, signature, jpeg artifacts, oversaturated, ugly"
)

# =====================================================================
# 6. GALERIE DE STYLES PREDEFINIS (BOUTONS RAPIDES)
# =====================================================================
# Chaque style ajoute une description anglaise optimisée pour SD1.5
# au prompt existant de l'utilisateur (ou le remplace si le prompt est vide).
STYLE_PRESETS = {
    "🎨 Aquarelle": "watercolor painting style, soft pastel colors, paper texture, delicate brush strokes",
    "🔷 Zellige": "intricate Moroccan zellige mosaic pattern, geometric tilework, vibrant blue and terracotta colors",
    "🌆 Cyberpunk": "cyberpunk style, neon lights, futuristic city, vibrant purple and cyan glow, cinematic",
    "🖌️ Huile Classique": "classical oil painting, renaissance style, rich textures, dramatic chiaroscuro lighting",
    "✏️ Croquis Crayon": "detailed pencil sketch, black and white, hand-drawn cross-hatching, fine line art",
    "🌸 Anime": "anime style illustration, vibrant cel-shaded colors, clean line art, studio ghibli inspired",
    "🏺 Patrimoine Marocain": "traditional Moroccan art style, intricate islamic geometric patterns, warm earthy palette",
    "🌌 Surréaliste": "surrealist painting style, dreamlike atmosphere, dali inspired, impossible architecture",
}

def apply_style_preset(style_name, current_prompt):
    """
    Construit le nouveau prompt en combinant le texte déjà tapé par
    l'utilisateur avec la description du style sélectionné.
    """
    style_description = STYLE_PRESETS.get(style_name, "")
    if not style_description:
        return current_prompt

    if current_prompt and current_prompt.strip():
        return f"{current_prompt.strip()}, {style_description}"
    return style_description

# =====================================================================
# 7. INTEGRATION GRADIO : FONCTIONS UTILITAIRES
# =====================================================================
def get_available_images(directory, extensions=("*.jpg", "*.jpeg", "*.png")):
    file_list = []
    for ext in extensions:
        file_list.extend(glob.glob(os.path.join(directory, ext)))
    return sorted([os.path.basename(f) for f in file_list])

def on_dataset_change(dataset_type):
    dir_path = "data/dataset_1_coco/val2017" if dataset_type == "Classique (COCO)" else "data/dataset_2_maroc"
    images = get_available_images(dir_path)
    if not images:
        return gr.Dropdown(choices=[], value=None, label="Sélectionner l'image - Aucun fichier trouvé")
    return gr.Dropdown(choices=images, value=images[0], label=f"Sélectionner l'image ({dataset_type})")

def update_image_preview(dataset_type, filename):
    if not filename: return None
    if dataset_type == "Classique (COCO)":
        base_dir = "data/dataset_1_coco/val2017"
    elif dataset_type == "Hard (Patrimoine Marocain)":
        base_dir = "data/dataset_2_maroc"
    else:
        base_dir = "data/dataset_3_wikiart"
    return os.path.join(base_dir, filename)

# CALLBACK NST
def process_nst_ui_flexible(content_dataset, content_file, content_upload, content_webcam, style_file, style_upload, num_steps, progress=gr.Progress()):
    if content_webcam is not None: content_path = content_webcam
    elif content_upload is not None: content_path = content_upload
    elif content_file:
        content_dir = "data/dataset_1_coco/val2017" if content_dataset == "Classique (COCO)" else "data/dataset_2_maroc"
        content_path = os.path.join(content_dir, content_file)
    else: return None, "Erreur Contenu", ""

    if style_upload is not None: style_path = style_upload
    elif style_file: style_path = os.path.join("data/dataset_3_wikiart", style_file)
    else: return None, "Erreur Style", ""

    progress(0, desc="Préparation des images...")
    preprocessor = MasterImagePreprocessor(target_size=(512, 512))
    content_tensor = preprocessor.preprocess_for_nst(content_path)
    style_tensor = preprocessor.preprocess_for_nst(style_path)

    generated_tensor = run_neural_style_transfer(content_tensor, style_tensor, int(num_steps), progress=progress)
    generated_pil = preprocessor.postprocess_to_pil(generated_tensor)
    original_pil = preprocessor.preprocess_for_diffusion(content_path)

    progress(1, desc="Calcul des métriques SSIM / LPIPS...")
    ssim_score, lpips_dist = compute_metrics(original_pil, generated_pil)

    return generated_pil, str(ssim_score), str(lpips_dist)

# CALLBACK DIFFUSION (Img2Img) - avec traduction multilingue + negative prompt + guidance
def process_diffusion_ui(content_dataset, content_file, content_upload, content_webcam,
                          prompt, strength, steps, guidance_scale, negative_prompt):
    if diffusion_pipe is None:
        return None, "Erreur : Pipeline non instancié ou manque de RAM/VRAM GPU.", ""

    if content_webcam is not None: content_path = content_webcam
    elif content_upload is not None: content_path = content_upload
    elif content_file:
        content_dir = "data/dataset_1_coco/val2017" if content_dataset == "Classique (COCO)" else "data/dataset_2_maroc"
        content_path = os.path.join(content_dir, content_file)
    else: return None, "Erreur Contenu", ""

    preprocessor = MasterImagePreprocessor(target_size=(512, 512))
    init_image = preprocessor.preprocess_for_diffusion(content_path)

    # 1. Traduction automatique du prompt (FR / AR / autres -> EN)
    prompt_en, detected_lang = translate_prompt_to_english(prompt)

    # 2. Enrichissement qualité
    final_prompt = enrich_prompt_for_quality(prompt_en)

    # 3. Negative prompt : celui de l'utilisateur (traduit si besoin) + défauts qualité
    if negative_prompt and negative_prompt.strip():
        neg_en, _ = translate_prompt_to_english(negative_prompt)
        final_negative_prompt = f"{neg_en}, {DEFAULT_NEGATIVE_PROMPT}"
    else:
        final_negative_prompt = DEFAULT_NEGATIVE_PROMPT

    with torch.autocast("cuda"):
        generated_pil = diffusion_pipe(
            prompt=final_prompt,
            negative_prompt=final_negative_prompt,
            image=init_image,
            strength=float(strength),
            num_inference_steps=int(steps),
            guidance_scale=float(guidance_scale)
        ).images[0]

    ssim_score, lpips_dist = compute_metrics(init_image, generated_pil)

    info_lang = f" (langue détectée: {detected_lang})" if detected_lang != "en" else ""
    return generated_pil, str(ssim_score), str(lpips_dist) + info_lang


# =====================================================================
# 8. GRADIO INTERFACE — DESIGN "FRESH" (palette claire turquoise / corail / menthe)
# =====================================================================
init_coco = get_available_images("data/dataset_1_coco/val2017")
init_styles = get_available_images("data/dataset_3_wikiart")
default_coco_val = init_coco[0] if init_coco else None
default_style_val = init_styles[0] if init_styles else None
default_coco_path = os.path.join("data/dataset_1_coco/val2017", default_coco_val) if default_coco_val else None
default_style_path = os.path.join("data/dataset_3_wikiart", default_style_val) if default_style_val else None

# ---------------------------------------------------------------------
# Palette "Fresh" :
#   - Turquoise/Teal  #14b8a6  (action principale)
#   - Corail          #fb7185  (accent secondaire / action diffusion)
#   - Menthe pâle     #ecfdf5  (fonds doux)
#   - Bleu ciel       #38bdf8  (liens / hover)
#   - Fond global très clair, presque blanc, légèrement bleuté
# ---------------------------------------------------------------------
custom_css = """
:root {
    --fresh-teal: #14b8a6;
    --fresh-teal-dark: #0d9488;
    --fresh-coral: #fb7185;
    --fresh-coral-dark: #f43f5e;
    --fresh-sky: #38bdf8;
    --fresh-mint-bg: #f0fdfa;
    --fresh-bg: #f7fafc;
    --fresh-card: #ffffff;
    --fresh-border: #d9f2ec;
    --fresh-text: #0f172a;
    --fresh-text-muted: #64748b;
}

.gradio-container {
    background: linear-gradient(180deg, #f7fdfb 0%, #f3fbf9 40%, #f7fafc 100%) !important;
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
    color: var(--fresh-text) !important;
}

/* ---------- Header ---------- */
.header-container {
    text-align: center;
    margin-bottom: 25px;
    padding: 30px 20px;
    background: linear-gradient(135deg, #14b8a6 0%, #38bdf8 55%, #5eead4 100%);
    border-radius: 18px;
    color: white;
    box-shadow: 0 8px 24px rgba(20, 184, 166, 0.25);
}
.header-container h1 {
    font-size: 2.3rem !important;
    font-weight: 800 !important;
    margin: 0;
    color: #ffffff;
    letter-spacing: -0.02em;
    text-shadow: 0 1px 6px rgba(0,0,0,0.08);
}
.header-container p {
    color: #ecfeff;
    margin-top: 8px;
    font-size: 1.05rem;
}

/* ---------- Tab nav (onglets principaux) ---------- */
.tab-nav button {
    border-radius: 10px 10px 0 0 !important;
    font-weight: 600 !important;
    color: var(--fresh-text-muted) !important;
}
.tab-nav button.selected {
    color: var(--fresh-teal-dark) !important;
    border-bottom: 3px solid var(--fresh-teal) !important;
    background: var(--fresh-mint-bg) !important;
}

/* ---------- Cards ---------- */
.section-card {
    border: 1px solid var(--fresh-border) !important;
    border-radius: 14px !important;
    padding: 18px !important;
    background: var(--fresh-card) !important;
    box-shadow: 0 2px 8px rgba(20, 184, 166, 0.06);
    margin-bottom: 16px;
    transition: box-shadow 0.2s ease, transform 0.15s ease;
}
.section-card:hover {
    box-shadow: 0 6px 16px rgba(20, 184, 166, 0.12);
}

.section-title {
    font-size: 1.05rem !important;
    font-weight: 700 !important;
    color: var(--fresh-text) !important;
    border-left: 4px solid var(--fresh-teal);
    padding-left: 10px;
    margin-bottom: 10px !important;
}

/* ---------- Inputs ---------- */
.gradio-container input, .gradio-container select, .gradio-container textarea {
    border-radius: 9px !important;
    border: 1px solid var(--fresh-border) !important;
    background: #fcfffe !important;
}
.gradio-container input:focus, .gradio-container textarea:focus {
    border-color: var(--fresh-teal) !important;
    box-shadow: 0 0 0 3px rgba(20, 184, 166, 0.15) !important;
}

/* ---------- Sliders ---------- */
input[type="range"] { accent-color: var(--fresh-teal) !important; }

/* ---------- Boutons principaux ---------- */
button.primary {
    background: linear-gradient(135deg, var(--fresh-teal), var(--fresh-sky)) !important;
    border: none !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 10px !important;
    box-shadow: 0 4px 12px rgba(20, 184, 166, 0.3);
}
button.primary:hover {
    filter: brightness(1.05);
    box-shadow: 0 6px 16px rgba(20, 184, 166, 0.4);
}
button.secondary {
    background: var(--fresh-mint-bg) !important;
    border: 1px solid var(--fresh-teal) !important;
    color: var(--fresh-teal-dark) !important;
    border-radius: 10px !important;
}

/* Bouton diffusion : variante corail pour différencier visuellement des deux modes */
#run_btn_diff {
    background: linear-gradient(135deg, var(--fresh-coral), var(--fresh-coral-dark)) !important;
    box-shadow: 0 4px 12px rgba(244, 63, 94, 0.3) !important;
}
#run_btn_diff:hover {
    box-shadow: 0 6px 16px rgba(244, 63, 94, 0.4) !important;
}
#run_btn_nst {
    background: linear-gradient(135deg, var(--fresh-teal), var(--fresh-sky)) !important;
}

/* ---------- Metric boxes ---------- */
.metric-box {
    background-color: var(--fresh-mint-bg) !important;
    border: 1px solid var(--fresh-border) !important;
    color: var(--fresh-teal-dark) !important;
    font-weight: 700 !important;
    text-align: center;
    border-radius: 10px !important;
}

/* ---------- Boutons "styles rapides" ---------- */
.style-preset-row { gap: 6px !important; flex-wrap: wrap; }
.style-preset-row button {
    border-radius: 20px !important;
    font-size: 0.85rem !important;
    padding: 6px 14px !important;
    border: 1px solid #99f6e4 !important;
    background: #f0fdfa !important;
    color: var(--fresh-teal-dark) !important;
    font-weight: 600 !important;
}
.style-preset-row button:hover {
    background: var(--fresh-teal) !important;
    border-color: var(--fresh-teal) !important;
    color: white !important;
}

/* ---------- Bandeau info langue ---------- */
.lang-hint {
    background: #f0f9ff;
    border: 1px solid #bae6fd;
    border-radius: 10px;
    padding: 8px 12px;
    font-size: 0.85rem;
    color: #0369a1;
    margin-bottom: 10px;
}

/* ---------- Image preview placeholders ---------- */
.section-card .image-container {
    background: var(--fresh-mint-bg) !important;
    border-radius: 12px !important;
    border: 1px dashed #99f6e4 !important;
}

#run_btn_nst, #run_btn_diff { font-weight: 700 !important; font-size: 1rem !important; padding: 11px !important; }
"""
FRAME_HEIGHT = 280

with gr.Blocks(theme=gr.themes.Soft(primary_hue="teal", secondary_hue="cyan"), css=custom_css) as demo:
    gr.HTML("""
    <div class='header-container'>
        <h1>🌿 Studio d'Architecture Générative Multi-Modèles</h1>
        <p>Basculez entre le Neural Style Transfer (VGG19 Optimization) et le Modèle Génératif (Stable Diffusion)</p>
    </div>
    """)

    with gr.Tabs() as main_tabs:

        # ==========================================
        # ONGLET 1 : NEURAL STYLE TRANSFER (NST)
        # ==========================================
        with gr.TabItem("🖼️ Mode : Neural Style Transfer", id="nst_tab"):

            # ---- Ligne 1 : Image de Contenu | Image de Style (côte à côte) ----
            with gr.Row():
                with gr.Column(scale=1):
                    with gr.Group(elem_classes="section-card"):
                        gr.Markdown("### 📂 1. Image de Contenu", elem_classes="section-title")
                        with gr.Tabs():
                            with gr.TabItem("📁 Sélection Dataset"):
                                content_dataset_nst = gr.Radio(choices=["Classique (COCO)", "Hard (Patrimoine Marocain)"], value="Classique (COCO)", label="Catégorie")
                                content_file_nst = gr.Dropdown(choices=init_coco, value=default_coco_val, label="Fichier")
                                content_preview_nst = gr.Image(value=default_coco_path, type="filepath", interactive=False, height=FRAME_HEIGHT)
                            with gr.TabItem("📤 Importer"):
                                content_upload_nst = gr.Image(type="filepath", height=FRAME_HEIGHT)
                            with gr.TabItem("📸 Webcam"):
                                content_webcam_nst = gr.Image(sources=["webcam"], type="filepath", height=FRAME_HEIGHT)

                with gr.Column(scale=1):
                    with gr.Group(elem_classes="section-card"):
                        gr.Markdown("### 🎭 2. Image de Style", elem_classes="section-title")
                        with gr.Tabs():
                            with gr.TabItem("🏛️ WikiArt"):
                                style_file_nst = gr.Dropdown(choices=init_styles, value=default_style_val, label="Style")
                                style_preview_nst = gr.Image(value=default_style_path, type="filepath", interactive=False, height=FRAME_HEIGHT)
                            with gr.TabItem("📤 Importer"):
                                style_upload_nst = gr.Image(type="filepath", height=FRAME_HEIGHT)

            # ---- Ligne 2 : Paramètres NST ----
            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### ⚙️ Paramètres NST", elem_classes="section-title")
                num_steps_nst = gr.Slider(minimum=50, maximum=500, step=50, value=100, label="Étapes d'optimisation (100 = bon compromis qualité/vitesse)")
                run_btn_nst = gr.Button("🚀 Exécuter la Stylisation (NST)", variant="primary", elem_id="run_btn_nst")

            # ---- Ligne 3 : Rendu NST & Métriques ----
            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### 🖼️ Rendu NST & Métriques", elem_classes="section-title")
                output_image_nst = gr.Image(label="Image Finale", type="pil", interactive=False, height=350)
                with gr.Row():
                    ssim_nst = gr.Textbox(label="Score SSIM", interactive=False, elem_classes="metric-box")
                    lpips_nst = gr.Textbox(label="Distance LPIPS", interactive=False, elem_classes="metric-box")

        # ==========================================
        # ONGLET 2 : STABLE DIFFUSION
        # ==========================================
        with gr.TabItem("🌀 Mode : Diffuser (Stable Diffusion)", id="diffusion_tab"):

            # ---- Ligne 1 : Structure de Base (Image Initiale) ----
            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### 📂 1. Structure de Base (Image Initiale)", elem_classes="section-title")
                with gr.Tabs():
                    with gr.TabItem("📁 Sélection Dataset"):
                        content_dataset_diff = gr.Radio(choices=["Classique (COCO)", "Hard (Patrimoine Marocain)"], value="Classique (COCO)", label="Catégorie")
                        content_file_diff = gr.Dropdown(choices=init_coco, value=default_coco_val, label="Fichier")
                        content_preview_diff = gr.Image(value=default_coco_path, type="filepath", interactive=False, height=FRAME_HEIGHT)
                    with gr.TabItem("📤 Importer"):
                        content_upload_diff = gr.Image(type="filepath", height=FRAME_HEIGHT)
                    with gr.TabItem("📸 Webcam"):
                        content_webcam_diff = gr.Image(sources=["webcam"], type="filepath", height=FRAME_HEIGHT)

            # ---- Ligne 2 : Paramètres & Conditionnement Sémantique ----
            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### 🎛️ Paramètres & Conditionnement Sémantique", elem_classes="section-title")
                gr.HTML("<div class='lang-hint'>✅ Le prompt peut être écrit en Français, Arabe ou Anglais — il sera traduit automatiquement.</div>")

                gr.Markdown("**🎨 Styles rapides** (clique pour ajouter au prompt) :")
                with gr.Row(elem_classes="style-preset-row"):
                    style_buttons = [gr.Button(name, size="sm") for name in STYLE_PRESETS.keys()]

                prompt_diff = gr.Textbox(
                    label="Prompt Artistique (Guidage Textuel) — FR / AR / EN",
                    placeholder="ex: Une belle peinture à l'huile de l'architecture marocaine, très détaillée, couleurs vives...",
                    lines=3
                )
                negative_prompt_diff = gr.Textbox(
                    label="Prompt Négatif (optionnel) — ce que l'image ne doit PAS contenir",
                    placeholder="ex: flou, mauvaise qualité, déformé...",
                    lines=2
                )
                strength_diff = gr.Slider(minimum=0.1, maximum=0.9, step=0.05, value=0.6, label="Force de Transformation (Strength — conserve la structure si bas)")
                steps_diff = gr.Slider(minimum=15, maximum=50, step=5, value=25, label="Étapes de Diffusion Denoising")
                guidance_scale_diff = gr.Slider(minimum=1.0, maximum=15.0, step=0.5, value=7.5, label="Guidance Scale (fidélité au prompt)")
                run_btn_diff = gr.Button("🪄 Lancer la Diffusion Générative", variant="primary", elem_id="run_btn_diff")

            # ---- Ligne 3 : Rendu Diffusion & Métriques ----
            with gr.Group(elem_classes="section-card"):
                gr.Markdown("### 🔮 Rendu Diffusion & Métriques", elem_classes="section-title")
                output_image_diff = gr.Image(label="Image Diffusée", type="pil", interactive=False, height=350)
                with gr.Row():
                    ssim_diff = gr.Textbox(label="Score SSIM", interactive=False, elem_classes="metric-box")
                    lpips_diff = gr.Textbox(label="Distance LPIPS", interactive=False, elem_classes="metric-box")

    # =====================================================================
    # EVENT BINDINGS
    # =====================================================================
    content_dataset_nst.change(fn=on_dataset_change, inputs=content_dataset_nst, outputs=content_file_nst)
    content_file_nst.change(fn=update_image_preview, inputs=[content_dataset_nst, content_file_nst], outputs=content_preview_nst)
    style_file_nst.change(fn=lambda f: update_image_preview("WikiArt", f), inputs=style_file_nst, outputs=style_preview_nst)
    run_btn_nst.click(
        fn=process_nst_ui_flexible,
        inputs=[content_dataset_nst, content_file_nst, content_upload_nst, content_webcam_nst, style_file_nst, style_upload_nst, num_steps_nst],
        outputs=[output_image_nst, ssim_nst, lpips_nst]
    )

    content_dataset_diff.change(fn=on_dataset_change, inputs=content_dataset_diff, outputs=content_file_diff)
    content_file_diff.change(fn=update_image_preview, inputs=[content_dataset_diff, content_file_diff], outputs=content_preview_diff)

    # Boutons de styles rapides : chacun ajoute sa description au prompt existant
    for btn, style_name in zip(style_buttons, STYLE_PRESETS.keys()):
        btn.click(
            fn=lambda current_prompt, s=style_name: apply_style_preset(s, current_prompt),
            inputs=prompt_diff,
            outputs=prompt_diff
        )

    run_btn_diff.click(
        fn=process_diffusion_ui,
        inputs=[content_dataset_diff, content_file_diff, content_upload_diff, content_webcam_diff,
                prompt_diff, strength_diff, steps_diff, guidance_scale_diff, negative_prompt_diff],
        outputs=[output_image_diff, ssim_diff, lpips_diff]
    )

# Indispensable sur Colab : inline=True et share=True pour générer un lien public temporaire gradio.live
demo.launch(inline=True, share=True)

Téléchargement des images du Patrimoine Marocain...
-> Dataset 2 (Patrimoine Marocain) installé avec succès : 0 images !
Chargement unique du modèle VGG19 pour le NST...
-> VGG19 chargé et mis en cache.
Initialisation du pipeline Stable Diffusion Img2Img...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_img2img.StableDiffusionImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2267: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.ena

-> xformers non disponible, on continue sans (pas bloquant).
-> Pipeline Stable Diffusion chargé avec succès !
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa8919afa6cd585480.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
